In [1]:
import pandas as pd
from pathlib import Path

landing = Path("../data/landing")
files = sorted(landing.glob("*.xlsx"))
for f in files:
    print(f.name)

Product-Statistics-UPI-Product-statistics-upi-2024-25-Monthly.xlsx
Product-Statistics-UPI-Product-statistics-upi-2025-26-Monthly.xlsx
Product-Statistics-UPI-Product-statistics-upi-2026-27-Monthly.xlsx


In [2]:
df = pd.read_excel(files[0])
df

,Month,No. of Banks live on UPI,Volume (In Mn.),Value (In Cr.)
0,March-2025,661,"18,301.51\t","24,77,221.61"
1,February-2025,653,16106.19,"21,96,481.69"
2,January-2025,647,16996,"23,48,037.12"
3,December-2024,641,16730.01,"23,24,699.91"
4,November-2024,637,15482.02,"21,55,187.40"
5,October-2024,632,16584.97,"23,49,821.46"
6,September-2024,622,15041.75,"20,63,994.71"
7,August-2024,608,14963.05,"20,60,735.57"
8,July-2024,605,14435.55,"20,64,292.41"
9,June-2024,602,13885.14,"20,07,081.20"


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Month                     12 non-null     str  
 1   No. of Banks live on UPI  12 non-null     int64
 2   Volume (In Mn.)           12 non-null     str  
 3   Value (In Cr.)            12 non-null     str  
dtypes: int64(1), str(3)
memory usage: 516.0 bytes


In [4]:
df.iloc[0].tolist()

['March-2025', np.int64(661), '18,301.51\t', '24,77,221.61']

In [5]:
for f in files:
    temp = pd.read_excel(f)
    print(f.name, temp.shape)
    print(list(temp.columns))
    print(temp["Month"].tolist()[:3])
    print("---")

Product-Statistics-UPI-Product-statistics-upi-2024-25-Monthly.xlsx (12, 4)
['Month', 'No. of Banks live on UPI', 'Volume (In Mn.)', 'Value (In Cr.)']
['March-2025', 'February-2025', 'January-2025']
---
Product-Statistics-UPI-Product-statistics-upi-2025-26-Monthly.xlsx (12, 4)
['Month', 'No. of Banks live on UPI', 'Volume (In Mn.)', 'Value (In Cr.)']
['March-2026', 'February-2026', 'January-2026']
---
Product-Statistics-UPI-Product-statistics-upi-2026-27-Monthly.xlsx (5, 4)
['Month', 'No. of Banks live on UPI', 'Volume (In Mn.)', 'Value (In Cr.)']
['August-2026', 'July-2026', 'June-2026']
---


In [6]:
clean = df.copy()
clean.columns = ["month", "banks_live", "volume_mn", "value_cr"]

for col in ["banks_live", "volume_mn", "value_cr"]:
    clean[col] = clean[col].astype(str).str.replace(",", "").str.strip().astype(float)

clean["banks_live"] = clean["banks_live"].astype(int)
clean["month"] = pd.to_datetime(clean["month"], format="%B-%Y")
clean = clean.sort_values("month").reset_index(drop=True)

clean["avg_ticket_rs"] = (clean["value_cr"] * 1e7 / (clean["volume_mn"] * 1e6)).round(0)
clean

,month,banks_live,volume_mn,value_cr,avg_ticket_rs
0,2024-04-01,583,13303.99,1964464.52,1477.0
1,2024-05-01,598,14035.84,2044937.05,1457.0
2,2024-06-01,602,13885.14,2007081.20,1445.0
3,2024-07-01,605,14435.55,2064292.41,1430.0
4,2024-08-01,608,14963.05,2060735.57,1377.0
5,2024-09-01,622,15041.75,2063994.71,1372.0
6,2024-10-01,632,16584.97,2349821.46,1417.0
7,2024-11-01,637,15482.02,2155187.40,1392.0
8,2024-12-01,641,16730.01,2324699.91,1390.0
9,2025-01-01,647,16996.00,2348037.12,1382.0


In [7]:
import duckdb

con = duckdb.connect("../data/upi.duckdb", read_only=True)
result = con.sql("""
    SELECT month, fiscal_year, avg_ticket_rs, volume_mom_pct, volume_yoy_pct
    FROM upi_metrics
    ORDER BY month
""").df()
con.close()
result

,month,fiscal_year,avg_ticket_rs,volume_mom_pct,volume_yoy_pct
0,2024-04-01,FY2024-25,1477.0,NaN,NaN
1,2024-05-01,FY2024-25,1457.0,5.50,NaN
2,2024-06-01,FY2024-25,1445.0,-1.07,NaN
3,2024-07-01,FY2024-25,1430.0,3.96,NaN
4,2024-08-01,FY2024-25,1377.0,3.65,NaN
5,2024-09-01,FY2024-25,1372.0,0.53,NaN
6,2024-10-01,FY2024-25,1417.0,10.26,NaN
7,2024-11-01,FY2024-25,1392.0,-6.65,NaN
8,2024-12-01,FY2024-25,1390.0,8.06,NaN
9,2025-01-01,FY2024-25,1382.0,1.59,NaN
